# EDA: People Wikipedia Dataset Analysis

In [1]:
# Standard imports
from pathlib import Path
import os
import sys

# --- FIX ---
# The original path logic was complex. This is simpler and more reliable.
# We assume the notebook is in `nlp_clustering/notebooks`.
# We go up two levels to get to the `nlp_clustering` directory, then into `src`.
try:
    # This works when running from inside the `notebooks` directory
    project_root = Path.cwd().parent.parent
    src_path = project_root / 'src'
    if not src_path.exists():
        # A fallback for when the CWD is the project root
        project_root = Path.cwd()
        src_path = project_root / 'src'
    
    sys.path.insert(0, str(src_path))
    
    import nlp_clustering.eda_utils as eda_utils
    print(f"Successfully added '{src_path}' to system path.")

except (ImportError, IndexError) as e:
    print(f"ERROR: Could not set up the correct path. Make sure you are running this notebook from the 'nlp_clustering/notebooks' directory.")
    print(f"Details: {e}")


print('Project root:', project_root)
print('Data folder:', project_root / 'data')

ERROR: Could not set up the correct path. Make sure you are running this notebook from the 'nlp_clustering/notebooks' directory.
Details: No module named 'nlp_clustering'
Project root: C:\Users\youse\Downloads\ML2 project\nlp_clustering\notebooks
Data folder: C:\Users\youse\Downloads\ML2 project\nlp_clustering\notebooks\data


In [2]:
# Load the People Wikipedia dataset
try:
    texts, names, df = eda_utils.load_dataset(data_dir=str(root / 'data'))
    print('People Wikipedia: documents =', len(texts))
    print('File loaded from:', eda_utils.resolve_data_path('people_wiki.csv', data_dir=str(root / 'data')))
    
    # Display the first 5 rows to inspect the data
    display(df.head())
    
except Exception as e:
    print('Could not load People Wikipedia dataset:', e)
    df = None

Could not load People Wikipedia dataset: name 'eda_utils' is not defined


## Basic checks: shapes and missing values

In [3]:
if df is not None:
    print('Dataset shape:', df.shape)
    print('\nMissing values:')
    display(eda_utils.missing_stats(df))
else:
    print('Dataset not loaded, skipping checks.')

Dataset not loaded, skipping checks.


## Text length analysis

In [4]:
if df is not None:
    # Calculate and print statistics about text length
    length_stats = eda_utils.text_length_stats(df['text'].tolist())
    print('Text length statistics (number of characters):')
    print(f"- Mean: {length_stats['mean']:.2f}")
    print(f"- Median: {length_stats['median']:.2f}")
    print(f"- Min: {length_stats['min']}, Max: {length_stats['max']}")
    
    # Plot the distribution of text lengths
    eda_utils.plot_length_distribution(
        df['text'].tolist(), 
        out_path=str(root / 'outputs' / 'people_length_distribution.png'),
        title='Distribution of Document Lengths'
    )
else:
    print('Text length analysis skipped: dataset not loaded.')

Text length analysis skipped: dataset not loaded.


## Word frequency analysis and top words

In [5]:
if df is not None:
    # Compute the top 50 most frequent words in the corpus
    freqs = eda_utils.compute_word_freqs(df['text'].tolist(), top_n=50)
    
    print('Top 20 most frequent words:')
    print(freqs[:20])
    
    # Plot the top 20 words in a bar chart
    eda_utils.plot_top_words(
        freqs, 
        out_path=str(root / 'outputs' / 'people_top_words.png'), 
        top_n=20,
        title='Top 20 Most Frequent Words'
    )
else:
    print('Word frequency analysis skipped: dataset not loaded.')

Word frequency analysis skipped: dataset not loaded.


## Word cloud (optional)

In [6]:
# WordCloud is an optional but powerful visualization
# Install with `pip install wordcloud` if not already installed
try:
    if 'freqs' in locals() and freqs:
        print("Generating word cloud...")
        eda_utils.plot_wordcloud_from_freqs(
            freqs, 
            out_path=str(root / 'outputs' / 'people_wordcloud.png'),
            title='Word Cloud of People Wikipedia Dataset'
        )
        print("Word cloud saved to outputs/people_wordcloud.png")
    else:
        print('Word cloud skipped: word frequencies not available.')
except Exception as e:
    print(f'Word cloud generation failed: {e}')

Word cloud skipped: word frequencies not available.


## Save quick CSV samples
Save small CSVs for downstream experiments and inspection.

In [7]:
if df is not None:
    # Save a sample of the dataframe for quick inspection or sharing
    sample_df = df.sample(min(200, len(df)), random_state=42)
    sample_path = root / 'outputs' / 'people_wiki_sample.csv'
    sample_df.to_csv(sample_path, index=False)
    print(f'Saved a sample of {len(sample_df)} rows to {sample_path}')
else:
    print('CSV sample creation skipped: dataset not loaded.')

CSV sample creation skipped: dataset not loaded.


### Conclusion and Next Steps

This EDA has provided valuable insights into the People Wikipedia dataset. We've confirmed its structure, analyzed text lengths, and identified the most common terms.

**Key Takeaways:**
- The dataset is clean with no missing text values.
- Text lengths vary, which is typical for biographical summaries.
- The most frequent words (e.g., 'born', 'university', 'he', 'his') are common in biographical texts and will likely be handled during preprocessing (e.g., stopword removal).

**Next Steps:**
- **Preprocessing**: Apply cleaning, tokenization, and lemmatization from `src/nlp_clustering/preprocessing.py`.
- **Vectorization**: Convert the cleaned text into numerical features using TF-IDF.
- **Clustering**: Apply algorithms like KMeans to group similar documents.